In [1]:
# ==========================================================
# ACTIVIDAD 19. Modelo definitivo (k = 3) y primera evaluación
# de la generalización. El modelo se entrena SOLO con empresas
# y bloque de entrenamiento y se aplica, sin reentrenar, a todas
# las filas: empresas reservadas y bloques de validación y prueba.
# ==========================================================
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

SEMILLA = 42
K = 3
NOMBRES = {1: "Calma", 2: "Impulso", 3: "Estrés"}
MUESTRA_SILUETA = 20_000
VARIABLES = ["volatilidad_21", "cambio_volatilidad", "rendimiento_21", "caida_maximo_63", "volumen_anormal"]
COLUMNAS_Z = ["z_" + v for v in VARIABLES]

drive.mount("/content/drive")
RAIZ = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Entrenar el modelo definitivo
# ----------------------------------------------------------
df = pd.read_parquet(RAIZ / "01_datos" / "variables_preparadas.parquet")
for col in ["ticker", "grupo", "bloque"]:
    df[col] = df[col].astype(str)
es_ajuste = (df["grupo"] == "entrenamiento") & (df["bloque"] == "entrenamiento")
modelo = KMeans(n_clusters=K, n_init=10, random_state=SEMILLA).fit(df.loc[es_ajuste, COLUMNAS_Z].to_numpy())
print("Modelo entrenado con", es_ajuste.sum(), "filas | inercia:", round(modelo.inertia_))

# Renumerar por volatilidad (1 = menor) para que coincida con la Actividad 18
orden = np.argsort(modelo.cluster_centers_[:, 0])
renumerar = {int(antiguo): nuevo + 1 for nuevo, antiguo in enumerate(orden)}

# ----------------------------------------------------------
# 2. Asignar un régimen a TODAS las filas (sin reentrenar)
# ----------------------------------------------------------
df["regimen"] = pd.Series(modelo.predict(df[COLUMNAS_Z].to_numpy()), index=df.index).map(renumerar)
df["nombre_regimen"] = df["regimen"].map(NOMBRES)
df["celda"] = df["grupo"] + " / " + df["bloque"]
ORDEN_CELDAS = ["entrenamiento / entrenamiento", "entrenamiento / validacion", "entrenamiento / prueba",
                "reservada / entrenamiento", "reservada / validacion", "reservada / prueba"]

# ----------------------------------------------------------
# 3. ¿Cuántos días hay en cada régimen? (empresas / periodo)
# ----------------------------------------------------------
print("\n=== % DE FILAS POR RÉGIMEN EN CADA CELDA (empresas / periodo) ===")
print((pd.crosstab(df["celda"], df["nombre_regimen"], normalize="index") * 100)
      .reindex(index=ORDEN_CELDAS, columns=list(NOMBRES.values())))

# ----------------------------------------------------------
# 4. ¿Cada régimen conserva su significado fuera del entrenamiento?
# ----------------------------------------------------------
for variable in ["volatilidad_21", "rendimiento_21", "caida_maximo_63", "volumen_anormal"]:
    print(f"\n=== MEDIANA DE {variable.upper()} POR RÉGIMEN Y CELDA ===")
    print(df.pivot_table(index="celda", columns="nombre_regimen", values=variable, aggfunc="median")
          .reindex(index=ORDEN_CELDAS, columns=list(NOMBRES.values())))

# ----------------------------------------------------------
# 5. Calidad de la agrupación en cada celda (silueta sobre una muestra fija)
# ----------------------------------------------------------
print("\n=== SILUETA POR CELDA (muestra de 20.000 filas) ===")
for celda in ORDEN_CELDAS:
    datos = df[df["celda"] == celda]
    muestra = datos.sample(min(MUESTRA_SILUETA, len(datos)), random_state=SEMILLA)
    print(f"{celda:32s} {silhouette_score(muestra[COLUMNAS_Z], muestra['regimen']):.3f}")

# ----------------------------------------------------------
# 6. Guardar el modelo y los regímenes de todas las filas
# ----------------------------------------------------------
joblib.dump({"modelo": modelo, "renumerar": renumerar, "nombres": NOMBRES},
            RAIZ / "02_modelos" / "kmeans_k3.joblib")
salida = df[["ticker", "date", "grupo", "bloque", "regimen", "nombre_regimen"] + VARIABLES].copy()
for col in ["ticker", "grupo", "bloque", "nombre_regimen"]:
    salida[col] = salida[col].astype("category")
salida.to_parquet(RAIZ / "01_datos" / "regimenes_k3.parquet", index=False)
print("\nGuardado: 02_modelos/kmeans_k3.joblib y 01_datos/regimenes_k3.parquet -", len(salida), "filas")

Mounted at /content/drive
Modelo entrenado con 815829 filas | inercia: 2495254

=== % DE FILAS POR RÉGIMEN EN CADA CELDA (empresas / periodo) ===
nombre_regimen                 Calma  Impulso  Estrés
celda                                                
entrenamiento / entrenamiento 52.909   32.921  14.170
entrenamiento / validacion    57.859   34.427   7.714
entrenamiento / prueba        54.106   34.524  11.369
reservada / entrenamiento     52.342   32.653  15.004
reservada / validacion        56.800   33.702   9.498
reservada / prueba            52.846   34.879  12.274

=== MEDIANA DE VOLATILIDAD_21 POR RÉGIMEN Y CELDA ===
nombre_regimen                 Calma  Impulso  Estrés
celda                                                
entrenamiento / entrenamiento  0.307    0.409   0.807
entrenamiento / validacion     0.248    0.352   0.673
entrenamiento / prueba         0.281    0.408   0.733
reservada / entrenamiento      0.307    0.411   0.800
reservada / validacion         0.249    0.3